<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/Self-Supervised%20Learning%20Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Self-Supervised Learning: SimCLR Implementation
This notebook demonstrates Contrastive Learning to train a model without labels using the STL-10 or CIFAR-10 dataset.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models, datasets
from torch.utils.data import DataLoader
import numpy as np

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


### 1. Data Augmentation
SimCLR relies heavily on strong augmentations (Color Jitter, Gaussian Blur, etc.) to create 'positive' pairs.

In [2]:
class ContrastiveLearningViewGenerator(object):
    def __init__(self, base_transforms, n_views=2):
        self.base_transforms = base_transforms
        self.n_views = n_views

    def __call__(self, x):
        return [self.base_transforms(x) for i in range(self.n_views)]

def get_simclr_pipeline_transform(size, s=1):
    color_jitter = transforms.ColorJitter(0.8 * s, 0.8 * s, 0.8 * s, 0.2 * s)
    data_transforms = transforms.Compose([
        transforms.RandomResizedCrop(size=size),
        transforms.RandomHorizontalFlip(),
        transforms.RandomApply([color_jitter], p=0.8),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
    return data_transforms

# Load STL-10 (unlabeled subset is perfect for SSL)
train_dataset = datasets.STL10(root='./data', split='unlabeled', download=True,
                               transform=ContrastiveLearningViewGenerator(get_simclr_pipeline_transform(32)))
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last=True)

100%|██████████| 2.64G/2.64G [07:40<00:00, 5.73MB/s]


### 2. SimCLR Model Architecture
We use a ResNet-18 backbone and add a 2-layer MLP projection head.

In [3]:
class SimCLR(nn.Module):
    def __init__(self, base_model, out_dim):
        super(SimCLR, self).__init__()
        self.resnet_dict = {"resnet18": models.resnet18(weights=None),
                            "resnet50": models.resnet50(weights=None)}

        self.backbone = self.resnet_dict[base_model]
        dim_mlp = self.backbone.fc.in_features

        # Replace the last fc layer with an identity function
        self.backbone.fc = nn.Identity()

        # Add projection head
        self.projection_head = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, out_dim)
        )

    def forward(self, x):
        h = self.backbone(x)
        z = self.projection_head(h)
        return z

### 3. Contrastive Loss (NT-Xent)
Normalized Temperature-scaled Cross Entropy loss.

In [4]:
def info_nce_loss(features, batch_size, temperature=0.5):
    labels = torch.cat([torch.arange(batch_size) for i in range(2)], dim=0)
    labels = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    labels = labels.to(device)

    features = nn.functional.normalize(features, dim=1)

    similarity_matrix = torch.matmul(features, features.T)

    # discard the main diagonal from loss: we don't want to compare a sample to itself
    mask = torch.eye(labels.shape[0], dtype=torch.bool).to(device)
    labels = labels[~mask].view(labels.shape[0], -1)
    similarity_matrix = similarity_matrix[~mask].view(similarity_matrix.shape[0], -1)

    # select and combine multiple positives
    positives = similarity_matrix[labels.bool()].view(labels.shape[0], -1)

    # select only the negatives
    negatives = similarity_matrix[~labels.bool()].view(similarity_matrix.shape[0], -1)

    logits = torch.cat([positives, negatives], dim=1)
    labels = torch.zeros(logits.shape[0], dtype=torch.long).to(device)

    logits = logits / temperature
    return logits, labels

model = SimCLR(base_model="resnet18", out_dim=128).to(device)
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

### 4. Training Loop

In [5]:
epochs = 10
for epoch in range(epochs):
    top1_acc = 0
    total_loss = 0
    for images, _ in train_loader:
        images = torch.cat(images, dim=0).to(device)

        features = model(images)
        logits, labels = info_nce_loss(features, batch_size=128)
        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

KeyboardInterrupt: 